In [1]:
# %pip install scipy

In [2]:
import glob
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
# from data_parsing.utils import resize
import numpy as np
from scipy.interpolate import interp1d

def resize(X, length, axis=1):
    """Resize the temporal length using linear interpolation.
    X must be of shape (N,M,C) (channels last) or (N,C,M) (channels first),
    where N is the batch size, M is the temporal length, and C is the number
    of channels.
    If X is channels-last, use axis=1 (default).
    If X is channels-first, use axis=2.
    """
    length_orig = X.shape[axis]
    t_orig = np.linspace(0, 1, length_orig, endpoint=True)
    t_new = np.linspace(0, 1, length, endpoint=True)
    X = interp1d(t_orig, X, kind="linear", axis=axis, assume_sorted=True)(
        t_new
    )
    return X

DEVICE_HZ = 100  # Hz
WINDOW_SEC = 10  # seconds
WINDOW_OVERLAP_SEC = 0  # i'm doing no overlap as I probably don't really care about activity boundaries
WINDOW_LEN = int(DEVICE_HZ * WINDOW_SEC)  # device ticks
WINDOW_OVERLAP_LEN = int(DEVICE_HZ * WINDOW_OVERLAP_SEC)  # device ticks
WINDOW_STEP_LEN = WINDOW_LEN - WINDOW_OVERLAP_LEN  # device ticks
WINDOW_TOL = 0.01  # 1%
TARGET_HZ = 30  # Hz
TARGET_WINDOW_LEN = int(TARGET_HZ * WINDOW_SEC)


datafile = os.path.join("..", "input", "10376_T1_MM_25022022_acc_gyr.parquet")
# data = pd.read_parquet(datafile)

OUTDIR = os.path.join("..", "data", "mobd_clean")

C:\Users\Lucy\anaconda3\envs\ssl-wearables\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# data

In [4]:
# TODO: this seems find but might as well check how it is used below
def is_good_quality(w):
    """Window quality check"""

    if w.isna().any().any():
        print("missing values - window rejected")
        return False

    if len(w) != WINDOW_LEN:
        print(f"window length {len(w)}, expected length {WINDOW_LEN} - window rejected")
        return False

    # if len(w['annotation'].unique()) > 1:
    # return False

    # this is testing window length, consistency etc but I think I've already accounted for this
    # in my data cleaning and it's throwing an error so I'm getting rid of it :)
    # w_start, w_end = w.index[0], w.index[-1]
    # w_duration = w_end - w_start
    # # target_duration = pd.Timedelta(WINDOW_SEC, "s")
    # if np.abs(w_duration - target_duration) > WINDOW_TOL * target_duration:
    # return False

    return True

In [5]:
# annolabel = pd.read_csv(ANNOLABELFILE, index_col='annotation')
# I need t for timestamp? TODO: need to think on this cause I am doing subject/timestamp splits not subject?
X, Y, T, P, = (
    [],
    [],
    [],
    [],
)


In [6]:
# TODO update this for my datafile (or actually can maybe keep these the same but change below to match my input file)
# column_names = ["time",  "x", "y", "z", "timestamp", "pid", "code"]

In [7]:
def process_windows(datafile):
    columns = ["time_acc", "acc_x", "acc_y", "acc_z", "timestamp", "p_id", "overall_nep_status"]
    one_person_data_t = pd.read_parquet(
        datafile,
        columns=columns
    )
    one_person_data_t.index = range(1, len(one_person_data_t) + 1)
    pid = one_person_data_t["p_id"].max()
    X_list = []
    Y_list = []
    T_list = []
    P_list = []

    # return one_person_data_t, pid
    for i in range(0, len(one_person_data_t), WINDOW_STEP_LEN):
        w = one_person_data_t.iloc[i : i + WINDOW_LEN]

        if not is_good_quality(w):
            # print("window no good")
            continue

        x = w[["acc_x", "acc_y", "acc_z"]].values
        t = w["timestamp"].max()
        y = w["overall_nep_status"].max()


        X_list.append(x)
        Y_list.append(y)
        T_list.append(t)
        P_list.append(pid)


    # Convert to numpy arrays
    X = np.array(X_list)
    Y = np.array(Y_list)
    T = np.array(T_list)
    P = np.array(P_list)

    print(X.shape, Y.shape, T.shape, P.shape)

    # fixing unit to g
    X = X / 9.81
    # downsample to 30 Hz
    X = resize(X, TARGET_WINDOW_LEN)

    print("dataset made!")

    os.system(f"mkdir -p {OUTDIR}")
    np.save(os.path.join(OUTDIR, "X"), X)
    np.save(os.path.join(OUTDIR, "Y"), Y)
    np.save(os.path.join(OUTDIR, "time"), T)
    np.save(os.path.join(OUTDIR, "pid"), P)

    print(f"Saved in {OUTDIR}")
    print("X shape:", X.shape)
    print("Y distribution:", len(set(Y)))
    print(pd.Series(Y).value_counts())
    print("User distribution:", len(set(P)))
    print(pd.Series(P).value_counts())


In [8]:
datafile = os.path.join("..", "input", "10376_T1_MM_25022022_acc_gyr.parquet")



process_windows(datafile)



missing values - window rejected
window length 750, expected length 1000 - window rejected
(63995, 1000, 3) (63995,) (63995,) (63995,)
dataset made!
Saved in ..\data\mobd_clean
X shape: (63995, 300, 3)
Y distribution: 1
0    63995
dtype: int64
User distribution: 1
10376    63995
dtype: int64


In [9]:
# lets see what their data looks like
# x_path = os.path.join("..", "data", "UKBB", "ssl_downstream", "adl_30hz_clean", "X.npy")
#
# X = np.load(x_path)
#
# print(X.shape)

In [10]:
columns = ["time_acc", "acc_x", "acc_y", "acc_z", "timestamp", "p_id", "overall_nep_status"]
one_person_data_t = pd.read_parquet(datafile,columns=columns)
one_person_data_t

,time_acc,acc_x,acc_y,acc_z,timestamp,p_id,overall_nep_status
0,NaN,NaN,NaN,NaN,T1,10376,0
1,1.644538e+09,-2.327167,-8.944734,-3.725380,T1,10376,0
2,1.644538e+09,-2.327167,-8.944734,-3.725380,T1,10376,0
3,1.644538e+09,-2.317586,-8.963896,-3.763684,T1,10376,0
4,1.644538e+09,-2.279281,-8.935162,-3.763684,T1,10376,0
...,...,...,...,...,...,...,...
63996745,1.645142e+09,-1.723823,-9.002201,-3.830723,T1,10376,0
63996746,1.645142e+09,-1.685518,-9.040505,-3.830723,T1,10376,0
63996747,1.645142e+09,-1.685518,-9.040505,-3.830723,T1,10376,0
63996748,1.645142e+09,-1.704670,-9.011772,-3.830723,T1,10376,0


In [11]:
for i in range(3):
    w = one_person_data_t.iloc[i : i + WINDOW_LEN]
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
        print(w)
        # if not is_good_quality(w):
        #     print("window no good")
        #     continue

        # x = w[["acc_x", "acc_y", "acc_z"]].values
        # t = w["timestamp"].max()
        # y = w["overall_nep_status"].max()
        #
        #
        # X_list.append(x)
        # Y_list.append(y)
        # T_list.append(t)
        # P_list.append(pid)

         time_acc     acc_x      acc_y     acc_z timestamp   p_id  \
0             NaN       NaN        NaN       NaN        T1  10376   
1    1.644538e+09 -2.327167  -8.944734 -3.725380        T1  10376   
2    1.644538e+09 -2.327167  -8.944734 -3.725380        T1  10376   
3    1.644538e+09 -2.317586  -8.963896 -3.763684        T1  10376   
4    1.644538e+09 -2.279281  -8.935162 -3.763684        T1  10376   
5    1.644538e+09 -2.317586  -8.944734 -3.744532        T1  10376   
6    1.644538e+09 -2.308015  -8.983048 -3.754113        T1  10376   
7    1.644538e+09 -2.298434  -8.944734 -3.754113        T1  10376   
8    1.644538e+09 -2.288852  -8.935162 -3.782837        T1  10376   
9    1.644538e+09 -2.327167  -8.973467 -3.763684        T1  10376   
10   1.644538e+09 -2.317586  -8.906429 -3.801989        T1  10376   
11   1.644538e+09 -2.308015  -8.925581 -3.792418        T1  10376   
12   1.644538e+09 -2.288852  -8.944734 -3.763684        T1  10376   
13   1.644538e+09 -2.298434  -8.96